# Fine-tune a small model on A-S-FLC (Colab + Unsloth)

**Goal:** Teach **Qwen2.5-1.5B** (or similar) to emit valid **DecisionOutput** JSON (A-S-FLC + optional security fields).

**Data:** Use `asflc_chat_format.jsonl` from this repo (`training/dataset/`) or load from Hugging Face dataset `denialkhmbot/a-s-flc-decisions` after you run `python training/format_for_hf.py --all` and upload.

**Eval:** Hold out IDs listed in `training/eval_split.json` — do not train on those rows.

## 1. Install dependencies

In [ ]:
%%capture
# Unsloth Colab install (see https://github.com/unslothai/unsloth for latest)
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes datasets huggingface_hub

## 2. Login to Hugging Face (for pushing the adapter)

```python
from huggingface_hub import login
login()
```

In [ ]:
# from huggingface_hub import login
# login()

## 3. Load model + tokenizer (4-bit QLoRA)

Adjust `max_seq_length` if you OOM. For DecisionOutput JSON, **2048–4096** is usually enough.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096
dtype = None  # Auto
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

## 4. Add LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

## 5. Load dataset (chat format)

**Option A — from Hugging Face:**
```python
from datasets import load_dataset
ds = load_dataset("json", data_files="hf://datasets/denialkhmbot/a-s-flc-decisions/asflc_chat_format.jsonl", split="train")
```

**Option B — upload `asflc_chat_format.jsonl` to Colab** and load locally.

In [ ]:
from datasets import load_dataset

# Replace with your path or HF URL after merge upload:
ds = load_dataset("json", data_files="asflc_chat_format.jsonl", split="train")

# TODO: filter out eval_ids from training/eval_split.json
eval_ids = set()  # load JSON list here
ds = ds.filter(lambda ex: ex["id"] not in eval_ids)

## 6. Format with chat template + train (SFT)

Use TRL `SFTTrainer` with `formatting_func` that concatenates messages via `tokenizer.apply_chat_template`. See Unsloth docs for the exact snippet — it changes slightly by library version.

**Hyperparameters (starting point):** `max_steps=60`, `learning_rate=2e-4`, `per_device_train_batch_size=2`, `gradient_accumulation_steps=4`.

In [ ]:
# Placeholder — complete using current Unsloth + TRL SFTTrainer example from:
# https://github.com/unslothai/unsloth/wiki
print("Wire SFTTrainer here (formatting_func + training args).")

## 7. Save + push LoRA / merged model

```python
model.save_pretrained("lora_asflc")
tokenizer.save_pretrained("lora_asflc")
# Optional: merge and export GGUF for llama.cpp / mobile
```